# 115 Python intermediate - Adnotacje typów (`typing`)  v2.0

## Rozkład jazdy

1. 🔹 **Podstawy adnotacji typów** - składnia, parametry, zwracanie
2. 🔹 **Moduł `typing`** - `Optional`, `Union`, `Literal`, `Callable`
3. 🔹 **`TypeVar`, `Generic` i `Protocol`** - typy parametryczne
   🐍 Ćwiczenia do każdej sekcji

---

## 1. 🔹 Podstawy adnotacji typów

Adnotacje typów (type annotations) to opcjonalne wskazówki informujące
o typach zmiennych, parametrów i wartości zwracanych. Python ich
**nie egzekwuje w czasie wykonania** - służą narzędziom statycznej
analizy (mypy, pyright, IDE).

```python
def greet(name: str) -> str:   # parametr: str, zwraca: str
    return f'Hello, {name}'

age: int = 25                  # zmienna z adnotacją
```

Od Pythona 3.9 wbudowane typy kolekcji można pisać małą literą:
`list[int]`, `dict[str, int]`, `tuple[int, ...]`.

| Stara składnia (3.8-) | Nowa składnia (3.9+) |
|---|---|
| `List[int]` | `list[int]` |
| `Dict[str, int]` | `dict[str, int]` |
| `Tuple[int, str]` | `tuple[int, str]` |
| `Set[str]` | `set[str]` |

> 💡 Adnotacje są przechowywane w `__annotations__` i dostępne
> przez `typing.get_type_hints(func)`.

In [ ]:
# Podstawowa składnia adnotacji
def add(x: int, y: int) -> int:
    return x + y


def greet(name: str, times: int = 1) -> str:
    return (f'Hello, {name}! ' * times).strip()


def process_items(items: list[int]) -> dict[str, int]:
    return {
        'sum':   sum(items),
        'min':   min(items),
        'max':   max(items),
        'count': len(items),
    }


print(add(1, 2))
print(greet('Alice', 3))
print(process_items([3, 1, 4, 1, 5, 9]))

# adnotacje nie są egzekwowane - Python tego nie sprawdza!
print(add('hello', ' world'))  # działa, mimo adnotacji int

# ale narzędzia statyczne (mypy) to wykryją
print(add.__annotations__)

---

### 🐍 Ćwiczenia - podstawy adnotacji

**Ćw. 1.** Dodaj adnotacje typów do następujących funkcji:
`calculate_bmi(weight_kg, height_m)` -> float,
`get_initials(full_name)` -> str (np. 'John Doe' -> 'J.D.'),
`flatten_once(nested)` -> list (list list -> jedna lista).

**Ćw. 2.** Utwórz klasę `Point` z adnotowanymi polami `x: float`
i `y: float`. Dodaj metody `distance_to(other: Point) -> float`
i `midpoint(other: Point) -> Point`.

**Ćw. 3.** *(Trudniejsze)* Napisz funkcję `group_by(items, key_func)`
z pełnymi adnotacjami. Funkcja grupuje elementy listy wg wyniku
`key_func`. Użyj nowych składni 3.9+ bez importowania z `typing`.
# hint: typ callable: Callable[[T], K] lub po prostu pisz opis

In [ ]:
# Ćw. 1
def calculate_bmi(weight_kg: ..., height_m: ...) -> ...:
    return weight_kg / height_m ** 2


def get_initials(full_name: ...) -> ...:
    ...


def flatten_once(nested: ...) -> ...:
    ...


print(f'{calculate_bmi(70, 1.75):.1f}')    # 22.9
print(get_initials('John Doe'))             # J.D.
print(flatten_once([[1, 2], [3], [4, 5]])) # [1, 2, 3, 4, 5]

In [ ]:
# Ćw. 2
import math

class Point:
    x: float
    y: float

    def __init__(self, x: float, y: float) -> None:
        ...

    def distance_to(self, other: 'Point') -> float:
        ...

    def midpoint(self, other: 'Point') -> 'Point':
        ...

    def __repr__(self) -> str:
        return f'Point({self.x}, {self.y})'


p1 = Point(0, 0)
p2 = Point(3, 4)
print(p1.distance_to(p2))   # 5.0
print(p1.midpoint(p2))      # Point(1.5, 2.0)

In [ ]:
# Ćw. 3
def group_by(items: list, key_func) -> dict:
    result: dict = {}
    for item in items:
        key = key_func(item)
        ...
    return result


words = ['apple', 'ant', 'banana', 'bear', 'cherry', 'cat']
grouped = group_by(words, lambda w: w[0])
print(grouped)
# {'a': ['apple','ant'], 'b': ['banana','bear'], 'c': ['cherry','cat']}

---

## 2. 🔹 Moduł `typing`

Moduł `typing` dostarcza specjalne typy do bardziej precyzyjnych
adnotacji:

| Typ | Opis | Przykład |
|---|---|---|
| `Optional[T]` | T lub None | `Optional[str]` |
| `Union[A, B]` | A lub B | `Union[int, str]` |
| `Literal[v, ...]` | tylko te konkretne wartości | `Literal['GET', 'POST']` |
| `Any` | dowolny typ (pomija sprawdzanie) | zmienne dynamiczne |
| `Callable[[A,B],R]` | funkcja przyjmująca A,B zwracająca R | callback |
| `Sequence[T]` | dowolna sekwencja (list, tuple, str) | |
| `Mapping[K,V]` | dowolny slownik-podobny obiekt | |
| `Iterable[T]` | dowolne iterable | |

Od Pythona 3.10 `Union[X, Y]` mozna pisac jako `X | Y`.

> 💡 `Optional[T]` to skrót dla `Union[T, None]`. Używamy go,
> gdy parametr lub wynik może być `None`.

In [ ]:
from typing import Optional, Union, Callable, Any


def find_user(user_id: int) -> Optional[dict]:
    users = {1: {'name': 'Alice'}, 2: {'name': 'Bob'}}
    return users.get(user_id)   # zwraca dict lub None


def stringify(value: Union[int, float, str]) -> str:
    return str(value)


def apply_twice(func: Callable[[int], int], value: int) -> int:
    return func(func(value))


print(find_user(1))                       # {'name': 'Alice'}
print(find_user(99))                      # None
print(stringify(3.14))
print(apply_twice(lambda x: x * 2, 3))   # 12


# Python 3.10+ - skrócona składnia Union
def divide(a: int, b: int) -> float | None:
    if b == 0:
        return None
    return a / b

print(divide(10, 2))   # 5.0
print(divide(10, 0))   # None

### `Literal` - ograniczenie do konkretnych wartości

`Literal[v1, v2, ...]` mówi narzędziom statycznym, że parametr
lub wynik może przyjąć **tylko** wymienione wartości literalne.
Python tego nie sprawdza w runtime, ale mypy/pyright zgłosi błąd
przy przekazaniu innej wartości.

Typowe zastosowania:
- tryby otwierania pliku: `Literal['r', 'w', 'a', 'rb', 'wb']`
- metody HTTP: `Literal['GET', 'POST', 'PUT', 'DELETE']`
- kierunki: `Literal['asc', 'desc']`
- poziomy logowania: `Literal['DEBUG', 'INFO', 'WARNING', 'ERROR']`

> 💡 `Literal` jest lepszy od `str` tam, gdzie tylko kilka wartości
> jest poprawnych - IDE podpowie dostępne opcje i mypy wykryje
> literówki jak `'GETT'` czy `'read'` zamiast `'r'`.

In [ ]:
from typing import Literal


# zamiast pisać str - piszemy dokładnie co jest dozwolone
HttpMethod  = Literal['GET', 'POST', 'PUT', 'DELETE', 'PATCH']
SortOrder   = Literal['asc', 'desc']
LogLevel    = Literal['DEBUG', 'INFO', 'WARNING', 'ERROR']


def send_request(url: str, method: HttpMethod) -> str:
    return f'{method} {url}'


def sort_items(items: list, order: SortOrder = 'asc') -> list:
    return sorted(items, reverse=(order == 'desc'))


def log(level: LogLevel, message: str) -> None:
    print(f'[{level}] {message}')


print(send_request('https://api.example.com/users', 'GET'))
print(sort_items([3, 1, 4, 1, 5], 'desc'))
log('WARNING', 'Low disk space')


# Literal mozna laczyc z Optional i Union
def open_file(path: str, mode: Literal['r', 'w', 'a'] = 'r'):
    return open(path, mode, encoding='utf-8')


# Literal dziala tez z wartosciami int i bool
def set_verbosity(level: Literal[0, 1, 2]) -> None:
    labels = {0: 'silent', 1: 'normal', 2: 'verbose'}
    print(f'Verbosity: {labels[level]}')

set_verbosity(1)

---

### 🐍 Ćwiczenia - moduł typing

**Ćw. 4.** Napisz funkcję `safe_parse_int(text: str) -> Optional[int]`
zwracającą `int` lub `None` gdy konwersja się nie powiedzie.
Następnie `parse_list(texts) -> list[int]` filtrującą tylko poprawne.

**Ćw. 5.** Napisz funkcję `transform(data, func)` przyjmującą
listę wartości i funkcję transformującą. Użyj `Callable` w adnotacji.
Przetestuj na liście liczb i liście stringów.

**Ćw. 6.** Napisz funkcję `align_text(text, alignment)`, gdzie
`alignment` jest typu `Literal['left', 'center', 'right']` a
`width: int = 40`. Funkcja wyrównuje tekst i zwraca `str`.
Napisz też `make_border(style: Literal['single', 'double'])` zwracający
linię z `─` lub `═`.

**Ćw. 7.** *(Trudniejsze)* Napisz funkcję `deep_get(d, *keys)`
do bezpiecznego dostępu do zagnieżdżonych słowników.
`deep_get({'a': {'b': 1}}, 'a', 'b')` -> `1`,
`deep_get({'a': {}}, 'a', 'x')` -> `None`.
Użyj `Union[dict, Any]` i `Optional[Any]` w adnotacjach.

In [ ]:
# Ćw. 4
from typing import Optional

def safe_parse_int(text: str) -> Optional[int]:
    ...


def parse_list(texts: list[str]) -> list[int]:
    ...


print(safe_parse_int('42'))              # 42
print(safe_parse_int('abc'))             # None
print(parse_list(['1', '2', 'x', '4'])) # [1, 2, 4]

In [ ]:
# Ćw. 5
from typing import Callable, TypeVar

T = TypeVar('T')
R = TypeVar('R')

def transform(data: list[T], func: Callable[[T], R]) -> list[R]:
    ...


print(transform([1, 2, 3, 4], lambda x: x ** 2))
print(transform(['hello', 'world'], str.upper))

In [ ]:
# Ćw. 6
from typing import Literal

Alignment = Literal['left', 'center', 'right']
BorderStyle = Literal['single', 'double']

def align_text(text: str, alignment: Alignment, width: int = 40) -> str:
    ...


def make_border(style: BorderStyle, width: int = 40) -> str:
    ...


print(align_text('Hello', 'left'))
print(align_text('Hello', 'center'))
print(align_text('Hello', 'right'))
print(make_border('single'))
print(make_border('double'))

In [ ]:
# Ćw. 7
from typing import Any, Optional

def deep_get(d: dict, *keys: str) -> Optional[Any]:
    ...


data = {'user': {'name': 'Alice', 'address': {'city': 'Warsaw'}}}
print(deep_get(data, 'user', 'name'))             # Alice
print(deep_get(data, 'user', 'address', 'city'))  # Warsaw
print(deep_get(data, 'user', 'phone'))             # None
print(deep_get(data, 'missing', 'key'))            # None

---

## 3. 🔹 `TypeVar`, `Generic` i `Protocol`

`TypeVar` to zmienna typów (type variable) - pozwala pisać funkcje
i klasy generyczne, gdzie konkretny typ jest znany dopiero w miejscu
użycia.

`Generic[T]` to klasa bazowa dla klas parametrycznych.

`Protocol` (duck typing) opisuje interfejs przez **obecność metod**,
nie przez dziedziczenie. Klasa implementuje protokół jeśli ma
odpowiednie metody - bez jawnego dziedziczenia.

| Mechanizm | Sprawdzanie | Wymagane dziedziczenie |
|---|---|---|
| `ABC` | nominalne (po nazwie) | tak |
| `Protocol` | strukturalne (duck typing) | nie |

> 💡 `Protocol` jest odpowiednikiem interfejsów z Java/C#, ale
> działa przez duck typing - nie wymaga jawnego `implements`.

In [ ]:
from typing import TypeVar, Generic, Protocol

T = TypeVar('T')


# TypeVar - funkcja generyczna
def first(items: list[T]) -> T:
    """Zwraca pierwszy element listy."""
    return items[0]

print(first([1, 2, 3]))        # 1 (int)
print(first(['a', 'b', 'c']))  # 'a' (str)


# Generic - klasa generyczna
class Stack(Generic[T]):
    def __init__(self) -> None:
        self._items: list[T] = []

    def push(self, item: T) -> None:
        self._items.append(item)

    def pop(self) -> T:
        return self._items.pop()

    def peek(self) -> T:
        return self._items[-1]

    def __len__(self) -> int:
        return len(self._items)


stack: Stack[int] = Stack()
stack.push(1)
stack.push(2)
print(stack.pop(), len(stack))  # 2 1


# Protocol - duck typing
class Drawable(Protocol):
    def draw(self) -> str: ...


class Circle:
    def draw(self) -> str: return 'Circle'


class Square:
    def draw(self) -> str: return 'Square'


def render(shape: Drawable) -> str:
    return shape.draw()


# Circle i Square nie dziedziczą po Drawable - ale protokół działa
print(render(Circle()))  # 'Circle'
print(render(Square()))  # 'Square'

---

### 🐍 Ćwiczenia - TypeVar, Generic, Protocol

**Ćw. 8.** Napisz generyczną funkcję `last(items: list[T]) -> T`
i `zip_with(a, b, func)` łączącą dwie listy funkcją. Użyj TypeVar.

**Ćw. 9.** Utwórz generyczną klasę `Pair[T, U]` przechowującą
parę wartości różnych typów. Dodaj metodę `swap() -> Pair[U, T]`
i `map_first(func) -> Pair`.

**Ćw. 10.** *(Trudniejsze)* Zdefiniuj `Protocol` o nazwie `Comparable`
z metodami `__lt__` i `__eq__`. Napisz generyczną funkcję
`binary_search(items: list[T], target: T) -> int` (zwraca indeks
lub -1) wymagającą T: Comparable. Sprawdź na liście liczb i stringów.
# hint: użyj TypeVar('T', bound=Comparable)

In [ ]:
# Ćw. 8
from typing import TypeVar, Callable

T = TypeVar('T')
R = TypeVar('R')

def last(items: list[T]) -> T:
    ...


def zip_with(
    a: list[T], b: list[T],
    func: Callable[[T, T], R],
) -> list[R]:
    ...


print(last([1, 2, 3]))           # 3
print(last(['x', 'y', 'z']))     # 'z'
print(zip_with([1, 2, 3], [4, 5, 6], lambda a, b: a + b))
# [5, 7, 9]

In [ ]:
# Ćw. 9
from typing import TypeVar, Generic, Callable

T = TypeVar('T')
U = TypeVar('U')
R = TypeVar('R')

class Pair(Generic[T, U]):
    def __init__(self, first: T, second: U) -> None:
        ...

    def swap(self) -> 'Pair[U, T]':
        ...

    def map_first(self, func: Callable[[T], R]) -> 'Pair[R, U]':
        ...

    def __repr__(self) -> str:
        return f'Pair({self.first!r}, {self.second!r})'


p = Pair(1, 'hello')
print(p)                              # Pair(1, 'hello')
print(p.swap())                       # Pair('hello', 1)
print(p.map_first(lambda x: x * 10)) # Pair(10, 'hello')

In [ ]:
# Ćw. 10
from typing import TypeVar, Protocol, runtime_checkable

@runtime_checkable
class Comparable(Protocol):
    def __lt__(self, other) -> bool: ...
    def __eq__(self, other) -> bool: ...

T = TypeVar('T', bound=Comparable)

def binary_search(items: list[T], target: T) -> int:
    ...


print(binary_search([1, 3, 5, 7, 9, 11], 7))    # 3
print(binary_search([1, 3, 5, 7, 9, 11], 4))    # -1
print(binary_search(['apple', 'banana', 'cherry', 'date'], 'cherry'))
# 2